In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.widgets import Slider, Button
from matplotlib.colors import hsv_to_rgb
from PIL import Image


# for files in TrainingDataFolder = r'C:\Users\ICNLab\caiman_data\Training_Data'  ending in .npy set MASK_FILE= that path, then find the single png file whose date time matches the datetime in the npy file's name.  The npy file is called r'C:\Users\ICNLab\caiman_data\Training_Data\'

MASK_FILE = "XXXX.npy"
IMG_FILE = "YYY.png"

# ----------------------------
# Load data
# ----------------------------
masks = np.load(MASK_FILE)  # (H,W,N)
H, W, N = masks.shape

img = np.array(Image.open(IMG_FILE).convert("L"))
p1, p99 = np.percentile(img, (1, 99))
img = np.clip((img - p1) / (p99 - p1), 0, 1)

# ----------------------------
# Color handling
# ----------------------------
def generate_colors(n):
    return hsv_to_rgb(
        np.column_stack([
            np.linspace(0, 1, n, endpoint=False),
            np.ones(n),
            np.ones(n)
        ])
    )

colors = generate_colors(N)
ALPHA = 0.35

# ----------------------------
# Main viewer
# ----------------------------
fig, ax = plt.subplots()
ax.set_title("Cell Mask Viewer")
ax.imshow(img, cmap="gray")

overlay = np.zeros((H, W, 4))
im_overlay = ax.imshow(overlay)

hovered = None
create_mode = {"active": False}

# ----------------------------
# Overlay redraw
# ----------------------------
def redraw_overlay(highlight=None):
    overlay[:] = 0
    for i in range(masks.shape[2]):
        if masks[:, :, i].any():
            overlay[masks[:, :, i] > 0, :3] = colors[i]
            overlay[masks[:, :, i] > 0, 3] = ALPHA
    if highlight is not None:
        overlay[masks[:, :, highlight] > 0, 3] = 0.8
    im_overlay.set_data(overlay)
    fig.canvas.draw_idle()

redraw_overlay()

# ----------------------------
# Hover logic
# ----------------------------
def on_move(event):
    global hovered
    if event.inaxes != ax or create_mode["active"]:
        return
    x, y = int(event.xdata), int(event.ydata)
    if x < 0 or y < 0 or x >= W or y >= H:
        return
    hits = np.where(masks[y, x, :] > 0)[0]
    if len(hits):
        if hovered != hits[0]:
            hovered = hits[0]
            redraw_overlay(hovered)
    else:
        if hovered is not None:
            hovered = None
            redraw_overlay()

# ----------------------------
# Click logic
# ----------------------------
def on_click(event):
    global masks, colors
    if event.inaxes != ax:
        return

    x, y = int(event.xdata), int(event.ydata)

    if create_mode["active"]:
        # Create new empty mask
        new_mask = np.zeros((H, W), dtype=bool)
        masks = np.dstack([masks, new_mask])
        colors = generate_colors(masks.shape[2])
        idx = masks.shape[2] - 1
        create_mode["active"] = False
        open_editor(idx, center=(y, x))
        redraw_overlay()
        return

    if hovered is not None:
        open_editor(hovered)

fig.canvas.mpl_connect("motion_notify_event", on_move)
fig.canvas.mpl_connect("button_press_event", on_click)

# ----------------------------
# New mask button
# ----------------------------
new_ax = plt.axes([0.01, 0.01, 0.18, 0.06])
new_btn = Button(new_ax, "New Mask")

def activate_new_mask(event):
    create_mode["active"] = True
    ax.set_title("Click anywhere to create a new mask")

new_btn.on_clicked(activate_new_mask)

# ----------------------------
# Editor window
# ----------------------------
def open_editor(idx, center=None):
    global masks

    mask = masks[:, :, idx]

    if center is None and mask.any():
        ys, xs = np.where(mask)
        cy, cx = int(np.mean(ys)), int(np.mean(xs))
    else:
        cy, cx = center if center else (H // 2, W // 2)

    r = 20
    y0, y1 = max(0, cy - r), min(H, cy + r)
    x0, x1 = max(0, cx - r), min(W, cx + r)

    sub_img = img[y0:y1, x0:x1]
    sub_mask = mask[y0:y1, x0:x1].copy()

    fig2, ax2 = plt.subplots()
    ax2.set_title(f"Editing mask {idx} (Draw)")
    ax2.imshow(sub_img, cmap="gray")
    mask_im = ax2.imshow(sub_mask, cmap="Reds", alpha=0.5)

    brush_ax = plt.axes([0.25, 0.02, 0.35, 0.03])
    brush_slider = Slider(brush_ax, "Brush", 1, 10, valinit=3)

    save_ax = plt.axes([0.63, 0.02, 0.15, 0.05])
    save_btn = Button(save_ax, "Save")

    del_ax = plt.axes([0.8, 0.02, 0.18, 0.05])
    del_btn = Button(del_ax, "Delete")

    mode = {"erase": False}

    def draw(event):
        if event.inaxes != ax2:
            return
        x, y = int(event.xdata), int(event.ydata)
        rr = int(brush_slider.val)
        yy, xx = np.ogrid[-rr:rr+1, -rr:rr+1]
        circle = xx**2 + yy**2 <= rr**2

        ys = slice(max(0, y-rr), min(sub_mask.shape[0], y+rr+1))
        xs = slice(max(0, x-rr), min(sub_mask.shape[1], x+rr+1))
        cm = circle[:ys.stop-ys.start, :xs.stop-xs.start]

        if mode["erase"]:
            sub_mask[ys, xs][cm] = 0
        else:
            sub_mask[ys, xs][cm] = 1

        mask_im.set_data(sub_mask)
        fig2.canvas.draw_idle()

    def toggle_mode(event):
        mode["erase"] = not mode["erase"]
        ax2.set_title(f"Editing mask {idx} ({'Erase' if mode['erase'] else 'Draw'})")

    def save(event):
        masks[y0:y1, x0:x1, idx] = sub_mask
        redraw_overlay()
        plt.close(fig2)

    def delete(event):
        nonlocal idx
        masks = np.delete(masks, idx, axis=2)
        redraw_overlay()
        plt.close(fig2)

    fig2.canvas.mpl_connect("motion_notify_event", draw)
    fig2.canvas.mpl_connect("button_press_event", toggle_mode)
    save_btn.on_clicked(save)
    del_btn.on_clicked(delete)

    plt.show()

plt.show()
